In [ ]:
import os
# os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--master local[3] pyspark-shell'
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql import Window

In [ ]:
spark = (
    SparkSession
    .builder
    .master('local')
    .config('spark.executor.memory', '5gb')
    .config("spark.cores.max", "6")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/06 16:21:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


 # couriers_orders

 У компании по доставке еды есть БД в которой содержится таблица заказов пеших курьеров couriers_orders.parquet.

In [ ]:
filename = 'couriers_orders.parquet'
path = str(Path.cwd() / filename)
sdf = spark.read.parquet(path)

In [ ]:
sdf.show()

+-------------------+----------+--------+--------+-----------+
|               date|courier_id|order_id|distance|travel_time|
+-------------------+----------+--------+--------+-----------+
|2021-07-12 00:00:00|        10|       1|     1.9|      36.17|
|2021-07-02 00:00:00|         3|       2|    3.98|      21.34|
|2021-04-15 00:00:00|         6|       3|    3.98|      43.33|
|2021-07-16 00:00:00|        10|       4|    2.85|      14.01|
|2021-06-11 00:00:00|        10|       5|    4.89|      32.09|
|2021-04-21 00:00:00|         9|       6|    1.06|      18.17|
|2021-07-12 00:00:00|         1|       7|    0.58|      19.22|
|2021-07-31 00:00:00|         5|       8|    3.97|      20.11|
|2021-06-14 00:00:00|         4|       9|    4.13|      29.34|
|2021-06-27 00:00:00|         8|      10|    1.04|      12.56|
|2021-07-26 00:00:00|         1|      11|     1.7|      29.89|
|2021-07-09 00:00:00|         9|      12|    0.58|      35.59|
|2021-07-13 00:00:00|         6|      13|    1.82|     

In [ ]:
sdf.printSchema()

root
 |-- date: timestamp_ntz (nullable = true)
 |-- courier_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- distance: double (nullable = true)
 |-- travel_time: double (nullable = true)



 ## Вопрос №1.1:
 В конце каждого месяца компания выдает премию для своих курьеров, средняя скорость доставки за прошедший месяц которых больше средней скорости среди всех курьеров. Сколько курьеров получили премию за июнь 2021 года.

In [ ]:
date_range = ('2021-06-01', '2021-06-30')

def calc_general_mean(df):
    """Calculates mean into a 'general_mean' global variable. Has no side effect on original DataFrame.

    Args:
        df: PySpark DataFrame

    Returns:
        df: PySpark DataFrame
    """
    global general_mean

    target_column = 'ind_mean_speed'
    general_mean = (
        df
        .agg(
            sf.mean(target_column).alias('general_mean')
        )
        .collect()
        [0]
        ['general_mean']
    )
    return df

# Result
(
    sdf
    .withColumn(
        'travel_speed',
         sf.col('distance') / sf.col('travel_time')
    )
    .filter(
        sf.col('date').between(*date_range)
    )
    .groupby('courier_id')
    .agg(
        sf.mean(sf.col('travel_speed')).alias('ind_mean_speed')
    )
    .transform(calc_general_mean)
    .filter(
        sf.col('ind_mean_speed') >= general_mean
    )
    .agg(sf.count('courier_id'))
    .show()
)

# temp = (
#     sdf
#     .withColumn(
#         'travel_speed',
#          sf.col('distance') / sf.col('travel_time')
#     )
#     .filter(
#         sf.col('date').between(*date_range)
#     )
#     .groupby('courier_id')
#     .agg(
#         sf.mean(sf.col('travel_speed')).alias('mean_speed')
#     )
# )

# general_mean_speed = (
#     temp
#     .agg(
#         sf.mean('mean_speed').alias('general_mean_speed')
#     )
#     .collect()
#     [0]
#     ['general_mean_speed']
# )

# # Result
# (
#     temp
#     .filter(
#         sf.col('mean_speed') >= general_mean_speed
#     )
#     .agg(sf.count('courier_id'))
# ).show()

+-----------------+
|count(courier_id)|
+-----------------+
|                6|
+-----------------+



 ### Result: 6

 ## Вопрос №1.2 (используйте данные из предыдущего вопроса №1.1):
 Компания хочет понять, насколько равномерно курьеры работают в течение месяца. Для этого нужно найти ID курьера с наибольшей разницей между максимальной и минимальной средней дневной скоростью в июне 2021 года.

In [ ]:
(
    sdf
    .withColumn(
        'travel_speed',
         sf.col('distance') / sf.col('travel_time')
    )
    .filter(
        sf.col('date').between(*date_range)
    )
    .groupby('courier_id', sf.day('date'))
    .agg(
        sf.mean('travel_speed').alias('day_mean_speed')
    )
    .groupby('courier_id')
    .agg(
       sf.min('day_mean_speed').alias('min_day_mean_speed'),
       sf.max('day_mean_speed').alias('max_day_mean_speed')
    )
    .withColumn(
        'diff',
         sf.col('max_day_mean_speed') - sf.col('min_day_mean_speed')
    )
    .sort(sf.desc('diff'))
).show()

+----------+--------------------+-------------------+-------------------+
|courier_id|  min_day_mean_speed| max_day_mean_speed|               diff|
+----------+--------------------+-------------------+-------------------+
|         4|0.016232668244842745|0.42960944595821987| 0.4133767777133771|
|         6|0.023384460648730197|  0.424953095684803| 0.4015686350360728|
|         1| 0.02775885558583106|0.42237222757955645|0.39461337199372537|
|         2| 0.02527981474334234|0.32341110217216407| 0.2981312874288217|
|        10|0.029186688022779853| 0.3100833965125095|0.28089670848972964|
|         5| 0.01840186372598687| 0.2941896024464831| 0.2757877387204963|
|         8|0.013209494324045407|0.24910265613783203| 0.2358931618137866|
|         9|0.012579762989972652|0.22192513368983954| 0.2093453706998669|
|         3|0.020884520884520884|0.22252131000448633|0.20163678911996544|
|         7|0.017559262510974536|0.18891815616180618|0.17135889365083165|
+----------+--------------------+-----

 ### Result: 4

 # purchases

 У нас есть данные о покупках клиентов purchases.parquet. Проанализируйте интервалы времени между последовательными покупками для каждого клиента в наборе данных о покупках - напишите код для вычисления разницы в днях между текущей покупкой и предыдущей покупкой каждого клиента. Отобразите результат в новом столбце days_between_purchases.

In [ ]:
filename = 'purchases.parquet'
path = str(Path.cwd() / filename)
sdf = spark.read.parquet(path)

In [ ]:
sdf.show()

+-----------+-------------------+
|customer_id|      purchase_date|
+-----------+-------------------+
|          2|2021-01-01 00:00:00|
|          7|2021-01-01 00:00:00|
|          7|2021-01-01 00:00:00|
|         11|2021-01-01 00:00:00|
|         21|2021-01-01 00:00:00|
|         22|2021-01-01 00:00:00|
|         27|2021-01-01 00:00:00|
|         30|2021-01-01 00:00:00|
|         36|2021-01-01 00:00:00|
|         45|2021-01-01 00:00:00|
|         50|2021-01-01 00:00:00|
|          3|2021-01-02 00:00:00|
|         15|2021-01-02 00:00:00|
|         18|2021-01-02 00:00:00|
|         24|2021-01-02 00:00:00|
|         34|2021-01-02 00:00:00|
|         35|2021-01-02 00:00:00|
|         40|2021-01-02 00:00:00|
|         41|2021-01-02 00:00:00|
|         12|2021-01-03 00:00:00|
+-----------+-------------------+
only showing top 20 rows


In [ ]:
sdf.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- purchase_date: timestamp_ntz (nullable = true)



In [ ]:
w = Window.partitionBy('customer_id').orderBy('purchase_date')
(
    sdf
    .withColumn('previous_purchase_date', sf.lag('purchase_date').over(w))
    .withColumn('days_between_purchases', sf.date_diff(sf.col('purchase_date'), sf.col('previous_purchase_date')))
    .show()
)

+-----------+-------------------+----------------------+----------------------+
|customer_id|      purchase_date|previous_purchase_date|days_between_purchases|
+-----------+-------------------+----------------------+----------------------+
|          1|2021-01-14 00:00:00|                  NULL|                  NULL|
|          1|2021-01-18 00:00:00|   2021-01-14 00:00:00|                     4|
|          1|2021-01-28 00:00:00|   2021-01-18 00:00:00|                    10|
|          1|2021-02-05 00:00:00|   2021-01-28 00:00:00|                     8|
|          1|2021-02-06 00:00:00|   2021-02-05 00:00:00|                     1|
|          1|2021-02-07 00:00:00|   2021-02-06 00:00:00|                     1|
|          1|2021-02-11 00:00:00|   2021-02-07 00:00:00|                     4|
|          1|2021-02-15 00:00:00|   2021-02-11 00:00:00|                     4|
|          1|2021-02-15 00:00:00|   2021-02-15 00:00:00|                     0|
|          1|2021-02-17 00:00:00|   2021

 ## Вопрос №2.1:
 Какое количество NaN в столбце days_between_purchases?

In [ ]:
w = Window.partitionBy('customer_id').orderBy('purchase_date')
(
    sdf
    .withColumn('previous_purchase_date', sf.lag('purchase_date').over(w))
    .withColumn('days_between_purchases', sf.date_diff(sf.col('purchase_date'), sf.col('previous_purchase_date')))
    .filter(sf.col('days_between_purchases').isNull())
    .count()
)

50

 ### Result: 50

 ## Вопрос №2.2 (используйте данные из предыдущего вопроса №2.1):
 У какого количества уникальных клиентов разница между текущей покупкой и предыдущей покупкой равна 20-ти дням?

In [ ]:
w = Window.partitionBy('customer_id').orderBy('purchase_date')
(
    sdf
    .withColumn('previous_purchase_date', sf.lag('purchase_date').over(w))
    .withColumn('days_between_purchases', sf.date_diff(sf.col('purchase_date'), sf.col('previous_purchase_date')))
    .filter(sf.col('days_between_purchases') == 20)
    .select(sf.col('customer_id'))
    .distinct()
    .count()
)

10

 ### Result: 10